<a href="https://colab.research.google.com/github/Realbenjizo/python-functions-for-data-engineers/blob/main/Chapter_7_Error_Handling_Patterns.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import logging

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

In [ ]:
import requests
import logging
from typing import Dict, Optional, TypedDict

logger = logging.getLogger(__name__)

def fetch_user_lookup(user_id) -> Dict[int, str]:
    """Fetch user data and handle expected errors gracefully."""
    try:
        # making an API call
        url = f"https://jsonplaceholder.typicode.com/users/{user_id}"
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        return response.json()
    except Exception:
        logger.exception("Unexpected error in fetching data for user_id=%s", user_id)
        raise

## **Better approach**

In [ ]:
from typing import Dict, Any
import requests
import logging

logger = logging.getLogger(__name__)

def fetch_user_lookup(user_id: int) -> Dict[str, Any]:
    """Fetch user data, using an empty fallback on timeout."""
    try:
        url = f"https://jsonplaceholder.typicode.com/users/{user_id}"
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        return response.json()

    except requests.exceptions.Timeout:
        # Recoverable locally: log a warning and return fallback
        logger.warning(f"User lookup API timed out for user_id {user_id}. Using empty fallback.")
        return {}

    except requests.exceptions.RequestException as e:
        # Only timeouts are recoverable. All other network/HTTP errors are fatal.
        raise RuntimeError(f"System failed to fetch profile for user {user_id}") from e

# Calling script
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO)
    try:
        user_data = fetch_user_lookup(5)
        print(user_data)
    except Exception:
        # The highest level of your code handles logging fatal errors exactly once
        logger.exception("Critical application breakdown while running user pipeline.")

{'id': 5, 'name': 'Chelsey Dietrich', 'username': 'Kamren', 'email': 'Lucio_Hettinger@annie.ca', 'address': {'street': 'Skiles Walks', 'suite': 'Suite 351', 'city': 'Roscoeview', 'zipcode': '33263', 'geo': {'lat': '-31.8129', 'lng': '62.5342'}}, 'phone': '(254)954-1289', 'website': 'demarco.info', 'company': {'name': 'Keebler LLC', 'catchPhrase': 'User-centric fault-tolerant solution', 'bs': 'revolutionize end-to-end systems'}}


# **Decorator for Consistent Error Handling**

In [ ]:
import functools
import logging
from typing import Callable, Any
import requests

logger = logging.getLogger(__name__)

def handle_api_exceptions(func: Callable[..., Any]) -> Callable[..., Any]:
    """Decorator to centralize logging and propagation of API failures."""
    @functools.wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        # Dynamically extract user_id from arguments for logging context
        user_id = kwargs.get("user_id") or (args[0] if args else "unknown")

        try:
            return func(*args, **kwargs)
        except requests.exceptions.Timeout:
            # Log warning and return a clean fallback
            logger.warning(
                f"API timeout in '{func.__name__}' for user_id {user_id}. "
                f"Returning empty fallback."
            )
            return {}
        except requests.exceptions.RequestException as e:
            # We do NOT log here to avoid duplicate log-and-rethrow noise.
            raise RuntimeError(
                f"System failure in '{func.__name__}' for user_id {user_id}"
            ) from e

    return wrapper

# Applying the decorator
@handle_api_exceptions
def fetch_user_lookup(user_id) -> Dict[int, str]:
    """Fetch user data and handle expected errors gracefully."""

    # making an API call
    url = f"https://jsonplaceholder.typicode.com/users/{user_id}"
    response = requests.get(url, timeout=5)
    response.raise_for_status()
    return response.json()

# Calling script
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO)
    try:
        user_data = fetch_user_lookup(5)
        logger.info(f"Successfully retrieved data:\n {user_data}")
    except Exception:
        logger.exception("Critical application breakdown while running user pipeline.")

INFO:__main__:Successfully retrieved data:
 {'id': 5, 'name': 'Chelsey Dietrich', 'username': 'Kamren', 'email': 'Lucio_Hettinger@annie.ca', 'address': {'street': 'Skiles Walks', 'suite': 'Suite 351', 'city': 'Roscoeview', 'zipcode': '33263', 'geo': {'lat': '-31.8129', 'lng': '62.5342'}}, 'phone': '(254)954-1289', 'website': 'demarco.info', 'company': {'name': 'Keebler LLC', 'catchPhrase': 'User-centric fault-tolerant solution', 'bs': 'revolutionize end-to-end systems'}}


# **Partial Failure handling full function with Decorator (Full function)**

In [ ]:
import functools
import logging
from typing import Dict, Any, List
import requests

logger = logging.getLogger(__name__)

# 2. Define the Exception-Handling Decorator
def handle_api_exceptions(func):
    """Decorator to centralize API error handling policies."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # Extract user_id dynamically for meaningful logs
        user_id = kwargs.get("user_id") or (args[0] if args else "unknown")

        try:
            return func(*args, **kwargs)

        except requests.exceptions.Timeout:
            # Recoverable locally: log a warning and return an empty fallback
            logger.warning(
                "API timeout in '%s' for user_id=%s. Using empty fallback.",
                func.__name__, user_id
            )
            return {}

        except requests.exceptions.RequestException as e:
            # We do NOT log here to avoid duplicate logs
            raise RuntimeError(f"API communication failure in '{func.__name__}'") from e

    return wrapper

# 3. Apply the Decorator to the Core API Call
@handle_api_exceptions
def fetch_user_lookup(user_id: int) -> Dict[str, Any]:
    """Fetch user data from external endpoint."""
    url = f"https://jsonplaceholder.typicode.com/users/{user_id}"
    response = requests.get(url, timeout=5)
    response.raise_for_status()
    return response.json()

# 4. Implement Partial Failure Handling
def enrich_users_with_limit(
    user_ids: List[int],
    max_failures: int = 2
) -> List[Dict[str, Any]]:
    """Fetch multiple users while tolerating a maximum number of failures."""
    results = []
    failed_ids = []

    for user_id in user_ids:
        try:
            user = fetch_user_lookup(user_id)
            # If the api timed out, the decorator safely returned {}
            if user:
                results.append(user)
            else:
                logger.info("Skipping user_id=%s due to local timeout fallback.", user_id)

        except RuntimeError as e:
            # Catch fatal exceptions wrapped and raised by the decorator
            failed_ids.append(user_id)
            current_failures = len(failed_ids)

            if current_failures > max_failures:
                raise RuntimeError(
                    f"Pipeline aborted. Failure threshold exceeded ({current_failures}/{max_failures}). "
                    f"Failed record IDs: {failed_ids}"
                ) from e

            # Log local warnings for records we choose to drop and skip
            logger.warning(
                "Skipping bad record user_id=%s. Current failure tolerance: %s/%s.",
                user_id, current_failures, max_failures
            )

    return results

# Calling Script
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO)

    target_users = [1, 2, 3]
    try:
        logger.info("Starting user enrichment pipeline...")
        results = enrich_users_with_limit(target_users, max_failures=2)
        logger.info(f"Pipeline completed successfully. Processed {len(results)} records.")
    except Exception:
        logger.exception("Critical pipeline execution failure. Process aborted.")

INFO:__main__:Starting user enrichment pipeline...
INFO:__main__:Pipeline completed successfully. Processed 3 records.


# **Context manager for Resource Clean-up**

### **1. Fragile implementation**

In [ ]:
import sqlite3
import pandas as pd
import logging

logger = logging.getLogger(__name__)

class DatabaseError(Exception):
    """Raised when a database operation fails."""

def save_to_db(df: pd.DataFrame, db_path: str, table_name: str) -> None:
    """Save DataFrame to database using manual resource cleanup."""
    conn = None
    try:
        conn = sqlite3.connect(db_path)
        df.to_sql(table_name, conn, if_exists="replace")
    except sqlite3.Error as e:
        raise DatabaseError(f"Database operation failed for table '{table_name}'") from e
    finally:
        # FRAGILE: Relies on Developer remembering to close connection
        if conn is not None:
            conn.close()

INFO:numexpr.utils:NumExpr defaulting to 2 threads.


### **Robust Implementation**

In [ ]:
class DatabaseError(Exception):
    """Raised when a database operation fails."""

def save_to_db(df: pd.DataFrame, db_path: str, table_name: str) -> None:
    """Save DataFrame to database using an automatic context manager."""
    try:
        with sqlite3.connect(db_path) as conn: # Using with statement
            df.to_sql(table_name, conn, if_exists="replace")
    except sqlite3.Error as e:
        # Error: Logged by global handler
        raise DatabaseError(
            f"Database operation failed for table {table_name} in {db_path}"
            ) from e

# Call the function
if __name__ == "__main__":
    logging.basicConfig(level=logging.ERROR)
    # Small DataFrame
    df = pd.DataFrame({
        'id': [1, 2, 3],
        'name': ['James', 'Bob', 'Lukas'],
        'age': [25, 30, 35]})
    try:
        save_to_db(df, "production.db", "users")
    except Exception:
        logger.exception("Catastrophic failure. Pipeline aborted.")

# **Using context manager decorator**

In [ ]:
import sqlite3
import logging
from contextlib import contextmanager
from typing import Generator
import pandas as pd

logger = logging.getLogger(__name__)

@contextmanager
def database_transaction(db_path: str) -> Generator[sqlite3.Connection, None, None]:
    """Provide database connection with automatic cleanup."""
    conn = sqlite3.connect(db_path)
    try:
        yield conn
        conn.commit()
    except Exception as e:
        # If an error occurs inside the block, roll back.
        conn.rollback()
        raise
    finally:
        conn.close()

# Applying context manager decorator
def save_to_db(df: pd.DataFrame, db_path: str, table_name: str) -> None:
    """Save user dataframe using our custom transaction manager."""
    # Reusing our custom manager handles connect, commit, rollback, and close
    with database_transaction(db_path) as conn:
        df.to_sql(table_name, conn, if_exists="replace", index=False)

# Execute script
if __name__ == "__main__":
    logging.basicConfig(level=logging.ERROR)
    # Small DataFrame
    df = pd.DataFrame({
        'id': [1, 2, 3],
        'name': ['James', 'Bob', 'Lukas'],
        'age': [25, 30, 35]})

    try:
        save_to_db(df, "production.db", table_name="users")
        logger.info("Data saved successfully")
    except Exception:
        # The top-level orchestrator to log and rollback history exactly once
        logger.exception("Pipeline crashed. All modifications safely rolled back.")

INFO:__main__:Data saved successfully


# **Challenge**

In [ ]:
from typing import List, Dict
import csv

def save_users_to_csv(users: List[Dict], filename: str) -> None:
    """Save user data to CSV file."""

    file = open(filename, 'w', newline='')
    writer = csv.DictWriter(file, fieldnames=['id', 'name', 'email'])
    writer.writeheader()
    for user in users:
        writer.writerow(user)
    file.close()

# Pass a list of dictionaries
details = [{"id": 101, "name": "Joe Doe", "email": "Ben@email.com"}]
save_users_to_csv(details, "trans.csv")

# **Challenge Solution**

In [ ]:
import csv
import sqlite3
import logging
from contextlib import contextmanager
from typing import List, Dict, Any, Generator

logger = logging.getLogger(__name__)

@contextmanager
def open_csv_file(filename: str, mode: str = 'w') -> Generator[sqlite3.Connection, None, None]:
    """Context manager to securely handle the lifecycle of a CSV file resource."""
    # Setup Phase: Open the file safely
    file_obj = open(filename, mode, newline='')
    try:
        yield file_obj
    except Exception as e:
        conn.rollback()
        raise
    finally:
        file_obj.close()

def save_users_to_csv(users: List[Dict[str, Any]], filename: str) -> None:
    """Save user data to a CSV file using our custom context manager."""
    # Using the context manager to hide connect/cleanup boilerplate
    with open_csv_file(filename, mode='w') as file:
        writer = csv.DictWriter(file, fieldnames=['id', 'name', 'email'])
        writer.writeheader()
        for user in users:
            writer.writerow(user)

# Execute the code
if __name__ == "__main__":
     # Configure logging
    logging.basicConfig(level=logging.ERROR)

    # Pass a list of dictionaries
    details = [{"id": 101, "name": "Joe Doe", "email": "Ben@email.com"}]
    try:
        save_users_to_csv(details, "trans.csv")
    except Exception:
        logger.exception("Pipeline crashed. All modifications rolled back.")